# Pipeline Dự Báo Giao Thông (utils + ml + rl)


Notebook này tổng hợp pipeline dự báo: nhận `request_time` từ app, lấy 12 timestep gần nhất, dự báo timestep tiếp theo (+15 phút), và chỉ giữ kết quả trong khung 09:15-21:15.

## Flow Chart Pipeline


```mermaid
flowchart TD
    A[App gửi request_time + corridor_id] --> B[Tải dữ liệu lịch sử bằng load_bulk_corridor_data]
    B --> C{Đủ 12 timestep liên tục?}
    C -- Không --> C1[Bỏ segment]
    C -- Có --> D[Tiền xử lý: encode + scale theo artifacts]
    D --> E[RL model suy luận Q-values]
    E --> F[Chọn lớp có Q-value lớn nhất]
    F --> G[Tính Forecast_For_Time = Window_End_Time + 15 phút]
    G --> H{Forecast_For_Time trong 09:15-21:15?}
    H -- Không --> H1[Loại khỏi kết quả]
    H -- Có --> I[Ghi output theo schema chuẩn]
    I --> J[Trả DataFrame/CSV/API response]
```


### Ghi chú nghiệp vụ


- `Window_End_Time` là mốc dữ liệu cuối dùng làm đầu vào.
- `Forecast_For_Time` mới là mốc thời gian được dự báo.
- Nếu App muốn dự báo mốc `t`, dữ liệu cần đủ để tạo cửa sổ kết thúc ở `t - 15 phút`.

## Input/Output Schema


### Input (từ App / API)


| Trường | Kiểu dữ liệu | Bắt buộc | Mô tả | Ví dụ |
|---|---|---|---|---|
| `corridor_id` | `int64` | Có | Mã corridor cần dự báo | `646713380690000556` |
| `request_time` | `datetime` | Có | Thời điểm App yêu cầu dự báo | `2026-04-08 20:00:00` |


### Xử lý nội bộ


- Truy vấn lịch sử dữ liệu đến `request_time`.
- Chọn 12 timestep liên tục gần nhất (chu kỳ 15 phút).
- Dự báo timestep kế tiếp bằng RL model.
- Lọc kết quả theo khung `09:15 - 21:15`.


### Output (trả về backend/app)


| Trường | Kiểu dữ liệu | Mô tả |
|---|---|---|
| `Segment_ID` | `int64` | Định danh đoạn đường |
| `Request_Time` | `datetime` | Thời điểm App gửi request |
| `Window_End_Time` | `datetime` | Mốc cuối của 12 timestep đầu vào |
| `Forecast_For_Time` | `datetime` | Mốc thời gian được dự báo (`Window_End_Time + 15m`) |
| `Dự báo (15p tới)` | `string` | Mức độ giao thông dự báo |
| `Q-Values (Kỳ vọng)` | `string/array` | Điểm kỳ vọng cho 6 mức lớp |

## Mục Tiêu Chức Năng


Pipeline này phục vụ nghiệp vụ dự báo giao thông theo yêu cầu thời gian thực từ App, với nguyên tắc:


- Khi nhận `request_time`, hệ thống tự truy xuất 12 timestep gần nhất cho từng segment.
- Dự báo mức ùn tắc cho timestep kế tiếp (`+15 phút`).
- Chỉ trả kết quả khi `Forecast_For_Time` nằm trong khung vận hành `09:15 - 21:15`.


### Kết quả mong muốn


- Dự báo ổn định theo từng segment trong corridor.
- Dễ tích hợp vào API backend với cấu trúc output rõ ràng.
- Tránh dự báo sai ngữ cảnh ngoài khung nghiệp vụ.

## 1) Thiết lập và import


- `utils.data_loader`: tải dữ liệu corridor


- `ml.traffic_model`: kiến trúc mô hình


- `rl.inference_rl`: predictor và pipeline theo request

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Đặt cwd về thư mục ai-core để import src.* ổn định
cwd = Path.cwd()
if cwd.name != 'ai-core':
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').exists() and p.name == 'ai-core':
            os.chdir(p)
            break

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Thư mục làm việc:', ROOT)

In [ ]:
from src.utils.data_loader import load_bulk_corridor_data
from src.ml.traffic_model import TrafficCongestionModel
from src.rl.inference_rl import (
    RLTrafficPredictor,
    forecast_for_request,
    _is_within_forecast_window,
)

print('Đã import thành công')

## 2) Cấu hình nghiệp vụ


Bạn có thể đổi `REQUEST_TIME` mỗi khi App gửi yêu cầu.


Quy tắc:


- Pipeline tự lấy 12 timestep gần nhất (`<= request_time`)


- Dự báo cho mốc tiếp theo (+15 phút)


- Chỉ nhận dự báo nếu `forecast_for_time` nằm trong 09:15-21:15

In [ ]:
# Cấu hình nghiệp vụ
CORRIDOR_ID = 646713380690000556
REQUEST_TIME = '2026-04-08 20:00:00'  # Thời điểm App gửi yêu cầu

MODEL_PATH = 'best_rl_agent.pt'
ARTIFACTS_PATH = 'preprocessing_artifacts.pkl'

print('Corridor:', CORRIDOR_ID)
print('Thời điểm request:', REQUEST_TIME)
print('Nằm trong khung dự báo:', _is_within_forecast_window(pd.to_datetime(REQUEST_TIME)))

## 3) Khởi tạo predictor (RL + ML artifacts)

In [ ]:
predictor = RLTrafficPredictor(
    model_path=MODEL_PATH,
    artifacts_path=ARTIFACTS_PATH,
)

print('Predictor đã sẵn sàng')
print('Lớp mô hình:', type(predictor.agent_net).__name__)
print('Thiết bị chạy:', predictor.device)

## 4) Chạy pipeline dự báo theo request


Ô này sẽ:


1. Gọi data_loader để lấy dữ liệu lịch sử corridor


2. Tự động cắt cửa sổ 12 timestep/segment


3. Dự báo +15 phút


4. Lọc theo khung 09:15-21:15

In [ ]:
df_results = forecast_for_request(
    predictor=predictor,
    corridor_id=CORRIDOR_ID,
    request_time=REQUEST_TIME,
    lookback_steps=12,
    resample_minutes=15,
)

print('Tổng số dự báo hợp lệ:', len(df_results))
df_results.head(10)

## 5) Tổng hợp nhanh kết quả

In [ ]:
if df_results.empty:
    print('Không có segment nào đạt điều kiện cho thời điểm request này.')
else:
    summary = (
        df_results.groupby('Dự báo (15p tới)').size()
        .sort_values(ascending=False)
        .rename('segment_count')
        .reset_index()
    )
    display(summary)

    print('Kiểm tra chân trời dự báo (phút):')
    horizon = (
        pd.to_datetime(df_results['Forecast_For_Time'])
        - pd.to_datetime(df_results['Window_End_Time'])
    ).dt.total_seconds().div(60)
    print(horizon.value_counts().sort_index())

## 6) Lưu output cho backend/app


File CSV có thể được backend đọc để trả về API.

In [ ]:
output_dir = Path('reports')
output_dir.mkdir(parents=True, exist_ok=True)

safe_request = pd.to_datetime(REQUEST_TIME).strftime('%Y%m%d_%H%M%S')
out_path = output_dir / f'rl_forecast_{CORRIDOR_ID}_{safe_request}.csv'

df_results.to_csv(out_path, index=False, encoding='utf-8-sig')
print('Đã lưu:', out_path)

## 7) Hàm wrapper để tích hợp với API App


Ô này đóng gói một hàm có thể gọi trực tiếp từ service layer.

In [ ]:
def run_request_pipeline(corridor_id: int, request_time: str) -> pd.DataFrame:
    predictor = RLTrafficPredictor(
        model_path='best_rl_agent.pt',
        artifacts_path='preprocessing_artifacts.pkl',
    )
    df = forecast_for_request(
        predictor=predictor,
        corridor_id=corridor_id,
        request_time=request_time,
        lookback_steps=12,
        resample_minutes=15,
    )
    return df

# Ví dụ:
# df_api = run_request_pipeline(646713380690000556, '2026-04-08 20:00:00')
# display(df_api.head())